# Flower Recommendation System for Classical Chinese Poetry

This notebook merges the current Flower Recommendation System pipeline into a single workflow, including data preprocessing, feature extraction, Qwen-based five-dimension classification, profile construction, and recommendation.

The latest version uses **Qwen-generated main labels plus semantic expansion labels** for user intent parsing. The recommender itself is responsible only for hard matching, soft matching, penalty handling, and ranking.

In [ ]:
import os
import re
import ast
import json
from collections import defaultdict
from typing import Dict, List

import jieba
import pandas as pd
import requests
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
import numpy as np

In [ ]:
# Path configuration (relative to notebook location in project2/ipynb/)
IPYNB_DIR = os.path.dirname(os.path.abspath('__file__')) if '__file__' in globals() else os.getcwd()
PROJECT2_DIR = os.path.dirname(IPYNB_DIR)
DATA_DIR = os.path.join(PROJECT2_DIR, 'data')
RESULT_DIR = os.path.join(PROJECT2_DIR, 'result')

POEM_CSV = os.path.join(DATA_DIR, 'poem.csv')
DICT_TXT = os.path.join(PROJECT2_DIR, 'dict.txt')
STOPWORDS_PATH = os.path.join(DATA_DIR, 'stopwords.txt')

POEMS_CLEAN_CSV = os.path.join(RESULT_DIR, 'poems_clean.csv')
FLOWER_CORPUS_CSV = os.path.join(RESULT_DIR, 'flower_corpus.csv')
FLOWER_TFIDF_CSV = os.path.join(RESULT_DIR, 'flower_tfidf_keywords.csv')
FLOWER_LDA_CSV = os.path.join(RESULT_DIR, 'flower_lda_keywords.csv')
FLOWER_FEATURE_POOL_CSV = os.path.join(RESULT_DIR, 'flower_feature_pool.csv')
FLOWER_PROFILES_JSON = os.path.join(RESULT_DIR, 'flower_profiles.json')

# Qwen API configuration
QWEN_API_KEY = 'sk-b02e6cf79dde4cc3b6763a8014b*****'
QWEN_BASE_URL = 'https://dashscope.aliyuncs.com/compatible-mode/v1'
QWEN_MODEL = 'qwen-turbo'

# Feature extraction parameters
TFIDF_TOP_N = 200
LDA_N_TOPICS = 20
LDA_TOP_N = 100
MAX_FEATURES = 3000

# Five semantic dimensions
FIVE_DIMS = ['entity', 'emotion', 'season', 'identity', 'occasion']

# Recommendation weights
DIM_WEIGHTS = {
    'emotion': 3.0,
    'occasion': 2.5,
    'identity': 3.5,
    'season': 1.5,
    'entity': 1.0,
}

SEASON_CONFLICT_PENALTY = 2.0
TOP_K = 3

os.makedirs(RESULT_DIR, exist_ok=True)

In [ ]:
def load_dim_guide(path=DICT_TXT):
    """Load the five-dimension guide dictionary from the plain text file."""
    with open(path, 'r', encoding='utf-8') as f:
        content = '{' + f.read().strip().strip(',') + '}'
    return ast.literal_eval(content)


def load_stopwords(path=STOPWORDS_PATH):
    """Load stopwords and append domain-specific low-information words."""
    stopwords = set()
    if os.path.exists(path):
        with open(path, 'r', encoding='utf-8') as f:
            for line in f:
                word = line.strip()
                if word:
                    stopwords.add(word)
    stopwords.update({
        '其', '之', '而', '亦', '更', '又', '一', '中', '下', '上', '一个', '一种',
        '可以', '使', '使得', '表现', '描写', '诗人', '诗中', '作品', '通过', '具有',
        '其中', '已经', '这个', '那个', '这些', '那些', '我们', '他们', '她们', '自己',
        '没有', '不是', '如果', '因此', '所以', '以及', '进行', '形成', '成为', '体现',
        '表达', '赏析', '正文', '这首', '两句', '词人', '感情', '二句', '三句', '人们',
        '作者', '正是', '写出', '艺术', '一首', '十分', '四句', '感受', '自然', '这里',
        '就是', '这种', '之中', '形象', '画面', '如此', '之情', '手法', '心理', '然而',
        '似乎', '看来', '这样', '这是', '此诗', '一句', '一片', '读者', '不同', '但是',
        '因为', '想象', '那么', '令人', '而是', '最后', '一样', '只有', '生活', '描绘'
    })
    return stopwords


def clean_text(text):
    """Normalize whitespace and remove punctuation."""
    text = str(text)
    text = re.sub(r'[\r\n\t]+', ' ', text)
    text = re.sub(r'[，。！？；：、“”‘’（）()《》【】\[\],.!?;:\-—…]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()


def tokenize_text(text, stopwords, dim_guide=None):
    """Tokenize Chinese text with jieba and preserve dimension guide words."""
    text = clean_text(text)
    tokens = []
    for word in jieba.lcut(text):
        word = word.strip()
        if not word:
            continue
        if len(word) <= 1:
            continue
        if word in stopwords:
            continue
        if re.fullmatch(r'\d+', word):
            continue
        tokens.append(word)

    if dim_guide:
        guide_words = {w for words in dim_guide.values() for w in words}
        for guide_word in guide_words:
            if guide_word in text and guide_word not in tokens:
                tokens.append(guide_word)
    return tokens

In [ ]:
def load_and_clean(path=POEM_CSV):
    """Load the structured poem dataset, standardize column names, and remove invalid rows."""
    df = pd.read_csv(path, encoding='utf-8')
    df = df.rename(columns={
        df.columns[3]: 'flower',
        df.columns[4]: 'poem_title',
        df.columns[5]: 'dynasty',
        df.columns[6]: 'author',
        df.columns[7]: 'content',
        df.columns[8]: 'analysis',
    })
    df = df[['flower', 'poem_title', 'dynasty', 'author', 'content', 'analysis']].copy()
    df = df.dropna(subset=['flower', 'content'])
    df['flower'] = df['flower'].astype(str).str.strip()
    df['content'] = df['content'].fillna('').astype(str).str.strip()
    df['analysis'] = df['analysis'].fillna('').astype(str).str.strip()
    df = df.drop_duplicates(subset=['flower', 'poem_title', 'content'])
    df.to_csv(POEMS_CLEAN_CSV, index=False, encoding='utf-8-sig')
    print(f'Cleaned dataset saved to: {POEMS_CLEAN_CSV}')
    return df


def build_flower_corpus(df):
    """Aggregate poem content and analysis at the flower level."""
    records = []
    for flower, group in df.groupby('flower'):
        combined_text = ' '.join((row['content'] + ' ' + row['analysis']) for _, row in group.iterrows())
        poem_count = len(group)
        rep = group.iloc[0]
        records.append({
            'flower': flower,
            'poem_count': poem_count,
            'corpus': combined_text,
            'rep_title': rep['poem_title'],
            'rep_dynasty': rep['dynasty'],
            'rep_author': rep['author'],
            'rep_content': rep['content'],
            'rep_analysis': rep['analysis'][:200],
        })
    corpus_df = pd.DataFrame(records)
    corpus_df.to_csv(FLOWER_CORPUS_CSV, index=False, encoding='utf-8-sig')
    print(f'Flower-level corpus saved to: {FLOWER_CORPUS_CSV}')
    return corpus_df

In [ ]:
def extract_tfidf_keywords(corpus_df, stopwords, dim_guide):
    """Extract flower-specific TF-IDF keywords from aggregated flower corpora."""
    token_docs = []
    for _, row in corpus_df.iterrows():
        tokens = tokenize_text(row['corpus'], stopwords, dim_guide)
        token_docs.append(' '.join(tokens))

    vectorizer = TfidfVectorizer(
        tokenizer=str.split,
        preprocessor=None,
        token_pattern=None,
        max_features=MAX_FEATURES,
    )
    tfidf_matrix = vectorizer.fit_transform(token_docs)
    feature_names = vectorizer.get_feature_names_out()

    records = []
    for idx, row in corpus_df.iterrows():
        vec = tfidf_matrix[idx].toarray().flatten()
        top_indices = vec.argsort()[::-1][:TFIDF_TOP_N]
        keywords = [feature_names[i] for i in top_indices if vec[i] > 0]
        records.append({
            'flower': row['flower'],
            'tfidf_keywords': ', '.join(keywords)
        })

    tfidf_df = pd.DataFrame(records)
    tfidf_df.to_csv(FLOWER_TFIDF_CSV, index=False, encoding='utf-8-sig')
    print(f'TF-IDF keywords saved to: {FLOWER_TFIDF_CSV}')
    return tfidf_df, token_docs


def extract_lda_keywords(corpus_df, token_docs):
    """Extract document-level LDA keywords for each flower instead of using shared global topic words."""
    vectorizer = CountVectorizer(
        tokenizer=str.split,
        preprocessor=None,
        token_pattern=None,
        max_features=MAX_FEATURES,
        min_df=1,
    )
    doc_term = vectorizer.fit_transform(token_docs)
    feature_names = vectorizer.get_feature_names_out()

    n_topics = min(LDA_N_TOPICS, max(2, len(corpus_df)))
    lda = LatentDirichletAllocation(
        n_components=n_topics,
        random_state=42,
        learning_method='batch',
        max_iter=30,
    )
    doc_topic = lda.fit_transform(doc_term)
    topic_word = lda.components_ / lda.components_.sum(axis=1, keepdims=True)

    records = []
    for idx, row in corpus_df.iterrows():
        doc_vector = doc_term[idx].toarray().flatten()
        doc_topic_dist = doc_topic[idx]
        word_scores = np.dot(doc_topic_dist, topic_word)
        combined_scores = word_scores * (doc_vector > 0)

        top_indices = combined_scores.argsort()[::-1]
        keywords = []
        for i in top_indices:
            if combined_scores[i] <= 0:
                continue
            word = feature_names[i]
            if word not in keywords:
                keywords.append(word)
            if len(keywords) >= LDA_TOP_N:
                break

        dominant_topic = int(doc_topic_dist.argmax())
        records.append({
            'flower': row['flower'],
            'lda_keywords': ', '.join(keywords),
            'dominant_topic': dominant_topic
        })

    lda_df = pd.DataFrame(records)
    lda_df.to_csv(FLOWER_LDA_CSV, index=False, encoding='utf-8-sig')
    print(f'LDA keywords saved to: {FLOWER_LDA_CSV}')
    return lda_df


def merge_feature_pool(tfidf_df, lda_df, dim_guide):
    """Merge TF-IDF and LDA features into a single candidate feature pool."""
    merged = tfidf_df.merge(lda_df, on='flower', how='left')
    guide_words = {w for words in dim_guide.values() for w in words}

    records = []
    for _, row in merged.iterrows():
        tfidf_words = [w.strip() for w in str(row['tfidf_keywords']).split(',') if w.strip()]
        lda_words = [w.strip() for w in str(row['lda_keywords']).split(',') if w.strip()]

        pool = []
        seen = set()
        for word in tfidf_words + lda_words:
            if word not in seen:
                pool.append(word)
                seen.add(word)

        guided_pool = [w for w in pool if w in guide_words]
        other_pool = [w for w in pool if w not in guide_words]
        final_pool = guided_pool + other_pool

        records.append({
            'flower': row['flower'],
            'feature_pool': ', '.join(final_pool[: max(TFIDF_TOP_N, LDA_TOP_N) * 2])
        })

    pool_df = pd.DataFrame(records)
    pool_df.to_csv(FLOWER_FEATURE_POOL_CSV, index=False, encoding='utf-8-sig')
    print(f'Feature pool saved to: {FLOWER_FEATURE_POOL_CSV}')
    return pool_df

In [ ]:
class QwenClassifier:
    """A lightweight wrapper for Qwen API to classify flower features and parse user intent."""

    def __init__(self, api_key: str = QWEN_API_KEY, model: str = QWEN_MODEL):
        self.api_key = api_key
        self.model = model
        self.base_url = QWEN_BASE_URL.rstrip('/')
        self.headers = {
            'Authorization': f'Bearer {self.api_key}',
            'Content-Type': 'application/json',
        }

    def _post_chat(self, system_prompt: str, user_prompt: str) -> str:
        payload = {
            'model': self.model,
            'messages': [
                {'role': 'system', 'content': system_prompt},
                {'role': 'user', 'content': user_prompt},
            ],
            'temperature': 0.1,
        }
        response = requests.post(
            f'{self.base_url}/chat/completions',
            headers=self.headers,
            data=json.dumps(payload),
            timeout=120,
        )
        response.raise_for_status()
        data = response.json()
        return data['choices'][0]['message']['content']

    @staticmethod
    def _extract_json(text: str) -> Dict:
        text = text.strip()
        match = re.search(r'```json\s*(\{.*?\})\s*```', text, re.S)
        if match:
            text = match.group(1)
        else:
            start = text.find('{')
            end = text.rfind('}')
            if start != -1 and end != -1 and end > start:
                text = text[start:end + 1]
        return json.loads(text)

    def classify_flower_features(self, flower: str, feature_words: List[str], dim_guide: Dict[str, List[str]]) -> Dict:
        """Classify candidate feature words into the five dimensions with stricter flower-specific constraints."""
        system_prompt = (
            'You are a Chinese classical poetry flower semantic classifier. '
            'You must return JSON only with no explanation and no markdown. '
            'Classify feature words into entity, emotion, season, identity, and occasion. '
            'The flower field must exactly match the input flower. '
            'The entity field can only contain the current flower itself, its aliases, close naming variants, or plant-part expressions associated with the same flower. '
            'Entity must never contain other flower categories. '
            'Words such as noble, lonely, elegant, sorrowful, pure, lovesick, homesick, youthful, late-life, reserved, and plaintive should be classified as emotion instead of unknown whenever reasonable. '
            'Season includes seasonal and climate words only. '
            'Identity includes human roles and relationship labels only. '
            'Occasion includes event or gifting scenarios only. '
            'Unknown should be minimized and used only when classification is truly uncertain. '
            'Each word can appear in at most one category.'
        )
        user_prompt = (
            f'Current flower: {flower}\n'
            f'Five-dimension guide: {json.dumps(dim_guide, ensure_ascii=False)}\n'
            f'Candidate feature words: {json.dumps(feature_words, ensure_ascii=False)}\n\n'
            'Return exactly this JSON structure:\n'
            '{"flower":"flower_name","entity":[],"emotion":[],"season":[],"identity":[],"occasion":[],"unknown":[]}'
        )
        raw = self._post_chat(system_prompt, user_prompt)
        result = self._extract_json(raw)
        for dim in FIVE_DIMS + ['unknown']:
            result.setdefault(dim, [])
        result['flower'] = flower
        return result

    def parse_user_input(self, user_input: str, dim_guide: Dict[str, List[str]]) -> Dict:
        """Parse user input into five dimensions and semantic expansion fields for soft retrieval."""
        system_prompt = (
            'You are a Chinese flower gifting intent parser. '
            'You must return JSON only with no explanation and no markdown. '
            'Extract entity, emotion, season, identity, and occasion from the user input. '
            'At the same time, you must provide semantic expansion for each dimension so that the recommender can use soft matching recall. '
            'If a dimension is not clearly expressed, return an empty list. '
            'Prefer classifying descriptive emotional adjectives and temperament words into emotion. '
            'Examples include noble, lonely, elegant, sorrowful, passionate, lovesick, homesick, farewell, remembrance, youth, and decline. '
            'You may do light semantic normalization such as teacher to mentor, winter to winter day, summer to summer day, girlfriend/wife/spouse to beloved, and father/mother/parents to elder. '
            'The identity dimension should prioritize kinship titles, human roles, and relationship labels. '
            'If the input contains father, mother, dad, mom, parents, elder family members, or senior-address terms, classify them into identity and normalize them toward elder whenever appropriate. '
            'If the input contains wife, spouse, partner, girlfriend, lover, or similar intimate-address terms such as wife, spouse, partner, girlfriend, lover, wife-address terms, bride-address terms, or affectionate partner titles, classify them into identity and normalize them toward beloved whenever appropriate. '
            'The semantic expansion must preserve the main label and then add close synonyms, related concepts, and useful hypernym or hyponym words for profile matching. '
            'For example, winter day may expand to severe cold, ice and snow, or cold season; mentor may expand to teacher, elder, gentleman, or master; elder may expand to father, mother, parents, senior, gentleman, or master; beloved may expand to wife, spouse, partner, girlfriend, lover, or beauty. '
            'Do not send obvious emotion words to unknown unless truly necessary.'
        )
        user_prompt = (
            f'Five-dimension guide: {json.dumps(dim_guide, ensure_ascii=False)}\n'
            f'User input: {user_input}\n\n'
            'Return exactly this JSON structure:\n'
            '{"entity":[],"emotion":[],"season":[],"identity":[],"occasion":[],"expanded":{"entity":[],"emotion":[],"season":[],"identity":[],"occasion":[]},"unknown":[]}'
        )
        raw = self._post_chat(system_prompt, user_prompt)
        result = self._extract_json(raw)
        for dim in FIVE_DIMS + ['unknown']:
            result.setdefault(dim, [])
        result.setdefault('expanded', {})
        for dim in FIVE_DIMS:
            result['expanded'].setdefault(dim, [])
        return result

In [ ]:
def build_profiles(feature_pool_csv=FLOWER_FEATURE_POOL_CSV, output_json=FLOWER_PROFILES_JSON):
    """Build flower profiles by classifying each flower's feature pool into five dimensions."""
    dim_guide = load_dim_guide()
    classifier = QwenClassifier()
    df = pd.read_csv(feature_pool_csv)
    profiles = []

    for _, row in df.iterrows():
        flower = row['flower']
        feature_words = [w.strip() for w in str(row['feature_pool']).split(',') if w.strip()]
        print(f'Building profile: {flower} ({len(feature_words)} candidate words)')
        result = classifier.classify_flower_features(
            flower=flower,
            feature_words=feature_words,
            dim_guide=dim_guide,
        )
        result['source_keywords'] = feature_words
        profiles.append(result)

    with open(output_json, 'w', encoding='utf-8') as f:
        json.dump(profiles, f, ensure_ascii=False, indent=2)

    print(f'Flower profiles saved to: {output_json}')
    return profiles

In [ ]:
class FlowerRecommender:
    """Recommend flowers by matching user intent against flower profiles and representative poems."""

    def __init__(self, profiles_path=FLOWER_PROFILES_JSON, corpus_path=FLOWER_CORPUS_CSV):
        self.classifier = QwenClassifier()
        self.dim_guide = load_dim_guide()
        with open(profiles_path, 'r', encoding='utf-8') as f:
            self.profiles = json.load(f)
        self.corpus_df = pd.read_csv(corpus_path)
        self.semantic_fallback = {
            'season': {
                '冬日': ['严寒', '冰雪', '岁寒', '寒香', '雪', '霜', '冬天'],
                '春天': ['早春', '春风', '报春', '东风'],
                '夏日': ['初夏', '暑气', '盛夏', '夏天'],
                '秋天': ['秋风', '金秋', '中秋', '重阳', '西风'],
            },
            'identity': {
                '师长': ['老师', '长辈', '君子', '先生', '师者'],
                '老师': ['师长', '长辈', '君子', '先生', '师者'],
                '长辈': ['父亲', '母亲', '爸爸', '妈妈', '家父', '家母', '父母', '双亲', '老人家', '长者', '尊者', '师长', '老师', '君子', '先生'],
                '父亲': ['长辈', '爸爸', '家父', '父母', '双亲', '先生', '君子'],
                '母亲': ['长辈', '妈妈', '家母', '父母', '双亲', '慈亲'],
                '父母': ['长辈', '父亲', '母亲', '双亲', '家父', '家母'],
                '朋友': ['知己', '故人', '友人'],
                '佳人': ['老婆', '妻子', '太太', '媳妇', '夫人', '爱妻', '内人', '对象', '伴侣', '女友', '女朋友', '恋人', '爱人', '美人'],
                '老婆': ['佳人', '妻子', '太太', '媳妇', '夫人', '爱妻', '内人', '女朋友', '恋人', '爱人', '美人'],
                '妻子': ['佳人', '老婆', '太太', '媳妇', '夫人', '爱妻', '内人'],
                '太太': ['佳人', '老婆', '妻子', '媳妇', '夫人', '爱妻', '内人'],
                '媳妇': ['佳人', '老婆', '妻子', '太太', '夫人', '爱妻', '内人'],
                '夫人': ['佳人', '老婆', '妻子', '太太', '媳妇', '爱妻', '内人'],
                '对象': ['佳人', '伴侣', '女友', '女朋友', '恋人', '爱人'],
                '伴侣': ['佳人', '对象', '恋人', '爱人', '女友', '女朋友'],
            },
            'emotion': {
                '高洁': ['清雅', '纯洁', '孤傲', '淡泊', '坚贞'],
                '寂寞': ['孤独', '凄凉', '幽怨', '清冷'],
                '怀念': ['思乡', '相思', '离别', '故园'],
            },
            'occasion': {
                '送别': ['送花', '赠别', '惜别'],
                '祝寿': ['贺寿', '敬寿'],
            },
        }

    def parse_user_intent(self, user_input):
        """Parse user input into structured five-dimension intent with semantic expansion."""
        result = self.classifier.parse_user_input(user_input, self.dim_guide)
        result = self._merge_semantic_fallback(result)
        print(f'[Parsed Intent] {result}')
        return result

    def _merge_semantic_fallback(self, intent):
        intent.setdefault('expanded', {})

        for dim in DIM_WEIGHTS.keys():
            raw_vals = intent.get(dim, []) or []
            expanded_vals = intent['expanded'].get(dim, []) or []
            merged_vals = []

            for value in raw_vals + expanded_vals:
                if value and value not in merged_vals:
                    merged_vals.append(value)

            fallback_dict = self.semantic_fallback.get(dim, {})
            for value in list(merged_vals):
                for fallback in fallback_dict.get(value, []):
                    if fallback not in merged_vals:
                        merged_vals.append(fallback)

            intent['expanded'][dim] = merged_vals

        return intent

    def _score_profile(self, intent, profile):
        """Compute weighted hard-match and soft-match scores between user intent and a flower profile."""
        score = 0.0
        reasons = defaultdict(list)

        expanded_intent = intent.get('expanded', {})

        for dim, weight in DIM_WEIGHTS.items():
            user_vals = set(intent.get(dim, []))
            expanded_user_vals = set(expanded_intent.get(dim, [])) | user_vals
            flower_vals = set(profile.get(dim, []))

            hard_hits = user_vals & flower_vals
            if hard_hits:
                score += len(hard_hits) * weight
                reasons[dim].extend(sorted(hard_hits))

            soft_hits = (expanded_user_vals & flower_vals) - hard_hits
            if soft_hits:
                score += len(soft_hits) * (weight * 0.6)
                reasons[f'{dim}_semantic'].extend(sorted(soft_hits))

        user_seasons = set(intent.get('season', []))
        expanded_user_seasons = set(expanded_intent.get('season', [])) | user_seasons
        flower_seasons = set(profile.get('season', []))
        if user_seasons and flower_seasons and not (expanded_user_seasons & flower_seasons):
            score -= SEASON_CONFLICT_PENALTY
            reasons['penalty'].append('season_conflict')

        feature_pool = set(profile.get('source_keywords', []))
        expanded_user_all = set()
        for dim in DIM_WEIGHTS.keys():
            expanded_user_all.update(intent.get(dim, []))
            expanded_user_all.update(expanded_intent.get(dim, []))
        weak_hits = expanded_user_all & feature_pool
        if weak_hits:
            score += 0.5 * len(weak_hits)
            reasons['feature_pool'].extend(sorted(weak_hits))

        return score, reasons

    def _get_poem_support(self, flower):
        """Retrieve the representative poem and full analysis for a flower."""
        row = self.corpus_df[self.corpus_df['flower'] == flower]
        if row.empty:
            return None
        row = row.iloc[0]
        return {
            'poem_title': str(row.get('rep_title', '')),
            'dynasty': str(row.get('rep_dynasty', '')),
            'author': str(row.get('rep_author', '')),
            'content': str(row.get('rep_content', '')),
            'analysis': str(row.get('rep_analysis', '')),
        }

    def recommend(self, user_input, top_k=TOP_K):
        """Run the full recommendation pipeline and return ranked flower candidates."""
        intent = self.parse_user_intent(user_input)
        results = []

        for profile in self.profiles:
            flower = profile.get('flower')
            score, reasons = self._score_profile(intent, profile)
            if score <= 0:
                continue

            support = self._get_poem_support(flower)
            results.append({
                'flower': flower,
                'score': round(score, 3),
                'reasons': dict(reasons),
                'support': support,
            })

        results = sorted(results, key=lambda x: x['score'], reverse=True)[:top_k]
        return results

    def pretty_print(self, user_input, top_k=TOP_K):
        """Format recommendation results into a readable output block with full poem content."""
        results = self.recommend(user_input, top_k=top_k)
        if not results:
            return 'No matching flower recommendation was found. Please refine your description.'

        lines = [f'\n🌸 Recommendation results for: {user_input}']
        for i, item in enumerate(results, start=1):
            lines.append(f"{i}. [{item['flower']}] Score: {item['score']}")
            lines.append(f"   Matched dimensions: {item['reasons']}")
            if item['support']:
                sup = item['support']
                lines.append(f"   Representative poem: {sup['poem_title']} | {sup['dynasty']} | {sup['author']}")
                lines.append(f"   Poem content: {sup['content']}")
                lines.append(f"   Analysis: {sup['analysis']}")
            lines.append('')
        return '\n'.join(lines)

## Recommendation Logic Summary

1. `QwenClassifier.parse_user_input()` outputs both **main labels** and **semantic expansion labels**, and now covers more common kinship and partner titles.
2. `FlowerRecommender` first merges parser output with a local `semantic_fallback` table to strengthen recall for common address terms.
3. It performs **hard matching** on the main labels.
4. It then performs **soft matching** on merged expanded labels with a discounted weight (`0.6 × dimension weight`).
5. A season conflict penalty is applied only when neither direct nor semantic season alignment exists.
6. A weak lexical overlap bonus is added if the user intent vocabulary overlaps with the flower source keyword pool.
7. Final ranking is based on the total weighted score, and representative poems are attached as evidence.

In [ ]:
# Example execution order
# df = load_and_clean()
# corpus_df = build_flower_corpus(df)
# stopwords = load_stopwords()
# dim_guide = load_dim_guide()
# tfidf_df, token_docs = extract_tfidf_keywords(corpus_df, stopwords, dim_guide)
# lda_df = extract_lda_keywords(corpus_df, token_docs)
# pool_df = merge_feature_pool(tfidf_df, lda_df, dim_guide)
# profiles = build_profiles()
# recommender = FlowerRecommender()
# print(recommender.pretty_print('Send flowers to a respected teacher in winter to express noble admiration'))